# Data Cleaning
This notebook is for cleaning the dataset in preperation for use in our various models.

## Import Packages

In [14]:
import pandas as pd
from missforest import MissForest

## Load Data

In [15]:
thyroid_df = pd.read_csv('Thyroid-Dataset.csv')
thyroid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9171 entries, 0 to 9170
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        9171 non-null   int64  
 1   sex                        8864 non-null   object 
 2   on thyroxine               9171 non-null   bool   
 3   query on thyroxine         9171 non-null   bool   
 4   on antithyroid medication  9171 non-null   bool   
 5   sick                       9171 non-null   bool   
 6   pregnant                   9171 non-null   bool   
 7   thyroid surgery            9171 non-null   bool   
 8   I131 treatment             9171 non-null   bool   
 9   query hypothyroid          9171 non-null   bool   
 10  query hyperthyroid         9171 non-null   bool   
 11  lithium                    9171 non-null   bool   
 12  goitre                     9171 non-null   bool   
 13  tumor                      9171 non-null   bool 

## Drop Referral Source Column

In [16]:
thyroid_df = thyroid_df.drop(columns='referral source')

## Drop Rows with Age Errors

In [17]:
age_error_rows = thyroid_df[thyroid_df['age'] > 100]
thyroid_df = thyroid_df.drop(age_error_rows.index)

## Create Missing Data Flags

### Float Variables

In [18]:
float_cols = thyroid_df.dtypes[thyroid_df.dtypes == 'float64'].index
for col in float_cols:
    new_col_name = col + '_missing'
    new_col_ind = thyroid_df.columns.to_list().index(col) + 1
    thyroid_df.insert(new_col_ind, new_col_name, thyroid_df[col].isna())

### Sex Variable

In [20]:
thyroid_df.insert(2, 'sex_missing', thyroid_df['sex'].isna())

## Dummify Class Variable

In [21]:
thyroid_df = pd.concat([thyroid_df, pd.get_dummies(thyroid_df['class'])], axis=1)
thyroid_df = thyroid_df.drop(columns='class')

## Imputation of Missing values

### Encode categorical and Boolean Variables

In [22]:
thyroid_df['sex'] = thyroid_df['sex'].map({'F': 0, 'M': 1}).astype('Int64')
bool_cols = thyroid_df.dtypes[thyroid_df.dtypes == 'bool'].index
for col in bool_cols:
    thyroid_df[col] = thyroid_df[col].map({True: 1, False: 0}).astype('int64')

### Impute Using MissForest Package

In [23]:

mf = MissForest(categorical=['sex'] + list(bool_cols))
thyroid_df = mf.fit_transform(thyroid_df)
print(type(thyroid_df))

d:\Programs\Python3.13\Lib\site-packages\missforest\missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
100%|██████████| 5/5 [00:13<00:00,  2.63s/it]
d:\Programs\Python3.13\Lib\site-packages\missforest\missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
d:\Programs\Python3.13\Lib\site-packages\missforest\missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting proce

<class 'pandas.core.frame.DataFrame'>


In [24]:
thyroid_df['sex'] = thyroid_df['sex'].astype('int64')
thyroid_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9167 entries, 0 to 9170
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        9167 non-null   int64  
 1   sex_missing                9167 non-null   int64  
 2   on thyroxine               9167 non-null   int64  
 3   query on thyroxine         9167 non-null   int64  
 4   sick                       9167 non-null   int64  
 5   on antithyroid medication  9167 non-null   int64  
 6   pregnant                   9167 non-null   int64  
 7   thyroid surgery            9167 non-null   int64  
 8   lithium                    9167 non-null   int64  
 9   I131 treatment             9167 non-null   int64  
 10  query hypothyroid          9167 non-null   int64  
 11  query hyperthyroid         9167 non-null   int64  
 12  tumor                      9167 non-null   int64  
 13  goitre                     9167 non-null   int64  
 1

### Save Cleaned Dataset

In [25]:
thyroid_df.to_csv('thyroid_cleaned.csv', index=False)